# STEP 0: Persiapan Environment
Instalasi dan import pustaka yang diperlukan untuk preprocessing teks.

In [ ]:
# Install library tambahan
!pip uninstall -y nltk Sastrawi
!pip install nltk Sastrawi swifter tqdm gensim sentence-transformers


In [ ]:
# Import pustaka
import pandas as pd
import re
import string
import nltk
import swifter
import os
import pickle
import numpy as np
import json
#import gdown
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# --- SETTINGS UNTUK TAMPILAN PANDAS ---
pd.set_option('display.max_columns', None)  # Tampilkan semua kolom
pd.set_option('display.width', 1000)      # Atur lebar tampilan di konsol
pd.set_option('display.max_rows', None)   # Tampilkan semua baris (gunakan hati-hati untuk DataFrame besar)
# ------------------------------------

tqdm.pandas() # Untuk progress bar di swifter

# Download resource NLTK
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

swifter.set_defaults(display_progress=True)

# STEP 1: Load Dataset

Di hands-on kedua ini, saya akan membuat sistem rekomendasi pekerjaan berdasarkan deskripsi course.

Jadi, sistem rekomendasi ini bisa digunakan untuk merekomendasikan kepada user untuk mendaftar ke sebuah pekerjaan setelah ia berhasil menyelesaikan sebuah course.

Dataset yang digunakan ada 2, yaitu:

## 1. Dealls Jobs Dataset
Dataset ini terdiri dari 4000+ baris data dengan isian berbagai kolom sebagai berikut:
- `Course Name`: Judul nama kursus.
- `Course Description`: Deskripsi detail isi kursus
- `Skills`: Kumpulan skill yang diajarkan dalam kursus

## 2. Coursera Courses Dataset
Dataset ini diambil dari kaggle yang memuat informasi kursus yang tersedia di Coursera, kolom yang digunakan hanya kolom nama kursus, deskripsi, dan skill yang diajarkan.

Kolom dataset:
- `Course Name`: Judul nama kursus.
- `Course Description`: Deskripsi detail isi kursus
- `Skills`: Kumpulan skill yang diajarkan dalam kursus

Sumber Dataset:
- Dealls Jobs Dataset = diperoleh dari hasil scrapping langsung menggunakan selenium.
- [Coursera Courses Dataset 2021](https://www.kaggle.com/datasets/khusheekapoor/coursera-courses-dataset-2021)

In [ ]:
# Set Up Folder untuk Simpan Model
MODEL_DIR = 'recommender_models'
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# Import Dataset Job
df_jobs = pd.read_csv('jobs_data.csv')
df_jobs.head()

In [ ]:
# Import Dataset Course
df_courses = pd.read_csv('courses_data.csv')
df_courses.head()

# STEP 2: Preprocessing
Di tahap ini, akan dilakukan pembersihan pada data. Untuk dataset jobs, akan dilakukan penggabungan pada kolom `Job Title`, `Job Description`, dan `Qualification` menjadi satu kolom, yaitu `Job Text`. Demikian pula untuk dataset course, akan diambil beberapa kolom yang mewakili informasi kursus, yaitu kolom `Title`, `Description`, dan `Skills` yang akan digabungkan menjadi satu kolom dengan nama `Course Text`.



In [ ]:
df_jobs['Job Text'] = df_jobs['Job Title'] +  ' ' + df_jobs['Job Description'] + ' ' + df_jobs['Qualification']
df_jobs.head()

In [ ]:
df_courses['Course Text'] = df_courses['Title'] +  ' ' + df_courses['Description'] + ' ' + df_courses['Skills']
df_courses.head()


Setelah itu akan dilakukan pembersihan data untuk masing - masing dataset dengan menghilangkan URL, simbol, angka, emoji, dan spasi berlebih. Dilakukan juga lowerisasi, tokenisasi, stopword removal, dan juga stemming.

In [ ]:
# Inisialisasi Stemmer
factory_stemmer = StemmerFactory()
stemmer = factory_stemmer.create_stemmer()

factory_stopword = StopWordRemoverFactory()
stopword_remover = factory_stopword.create_stop_word_remover()

stop_words_en = set(stopwords.words('english'))
stop_words_id = set([
    'yang', 'dan', 'di', 'ke', 'dari', 'untuk', 'dengan', 'atau', 'pada', 'ini', 'itu',
    'sebagai', 'jika', 'ada', 'karena', 'saat', 'oleh', 'juga', 'agar', 'dalam', 'tidak',
    'adalah', 'bahwa', 'maupun', 'bagi', 'dapat', 'telah', 'sudah', 'lebih', 'harus',
    'setiap', 'kita', 'kami', 'saya', 'anda', 'mereka', 'semua', 'bisa', 'masih',
    'akan', 'dan', 'pun', 'hingga', 'dengan', 'berikut', 'namun', 'sehingga'
])
stop_words_all = stop_words_en.union(stop_words_id)

In [ ]:
# Definisi Fungsi Preprocessing
def clean_noise(text):
    if not isinstance(text, str): # Pastikan input adalah string
        return ""
    text = re.sub(r'http\S+|www\S+', '', text)                       # hapus URL
    text = re.sub(r'\d+', '', text)                                  # hapus angka
    text = text.translate(str.maketrans('', '', string.punctuation)) # hapus tanda baca & simbol
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)                       # hapus emoji dan karakter aneh
    text = re.sub(r'\s+', ' ', text).strip()                         # trim spasi berlebih
    return text

def tokenisasi(text):
    return word_tokenize(text)

def stopword(tokens):
    return [word for word in tokens if word not in stop_words_all and len(word) > 2] # Filter kata < 3 huruf

def stemming(token_list):
    return [stemmer.stem(word) for word in token_list]

# Fungsi gabungan untuk preprocessing teks baru (pipeline)
def cleaning_pipeline(text):
    # Step 1: Bersihkan URL, simbol, angka, emoji
    text = clean_noise(text)

    # Step 2: Lowercase
    text = text.lower()

    # Step 3: Tokenisasi
    tokens = tokenisasi(text)

    # Step 4: Stopword Removal
    tokens = stopword(tokens)

    # Step 5: Stemming
    stemmed = stemming(tokens)

    return ' '.join(stemmed)


In [ ]:
df_jobs['Job Text'] = df_jobs['Job Text'].swifter.apply(cleaning_pipeline)
df_jobs.head()

In [ ]:
# Pecah kolom 'Work Mode' pada dataset Job menjadi dua kolom baru
df_jobs[['Work Arrangement', 'Location']] = df_jobs['Work Mode'].str.split(' ? ', expand=True)

# Bersihkan spasi berlebih
df_jobs['Work Arrangement'] = df_jobs['Work Arrangement'].str.strip()
df_jobs['Location'] = df_jobs['Location'].str.strip()

print("? Work Mode dipecah menjadi Work Arrangement dan Location:")
df_jobs[['Work Mode', 'Work Arrangement', 'Location']].head()


In [ ]:
# Simpan ke CSV supaya tidak perlu preprocess ulang selama data masih sama
df_jobs.to_csv('df_jobs_cleaned.csv', index=False)
print("? Preprocessing data job selesai, data disimpan ke df_jobs_cleaned.csv")


In [ ]:
df_courses['Course Text'] = df_courses['Course Text'].swifter.apply(cleaning_pipeline)
df_courses.head()

In [ ]:
# Simpan ke CSV supaya tidak perlu preprocess ulang selama data masih sama
df_courses.to_csv('df_courses_cleaned.csv', index=False)
print("? Preprocessing data course selesai, data disimpan ke df_courses_cleaned.csv")


In [ ]:
# Contoh input
new_text = "Dicari kandidat yang memiliki pengalaman data engineering minimal 2 tahun dan siap kerja remote."

# Preprocess
cleaned_tokens = cleaning_pipeline(new_text)

print("Teks awal :", new_text)
print("\nTeks akhir:", cleaned_tokens)


# STEP 3: Data Vectorization

## TF-IDF

In [ ]:
# Inisialisasi TF-IDF Vectorizer

# max_features: Batas jumlah kata unik yang digunakan sebagai fitur.
#               Membantu mengurangi dimensi dan fokus pada kata-kata paling penting.
tfidfv = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)

In [ ]:
# Fill NaN values in 'Job Text' with empty strings before fitting
df_jobs['Job Text'] = df_jobs['Job Text'].fillna('')

# Fit dan transform teks job menjadi vektor
job_vectors = tfidfv.fit_transform(df_jobs['Job Text'])

# Menampilkan fitur (kata unik)
print("Fitur (Kata unik):")
print(tfidfv.get_feature_names_out())

# Menampilkan matriks fitur dalam bentuk array
print("\nMatriks TF-IDF:")
print(job_vectors.toarray())


In [ ]:
# Fill NaN values in 'Course Text' with empty strings before fitting
df_courses['Course Text'] = df_courses['Course Text'].fillna('')

# Fit dan transform teks course menjadi vektor
course_vectors = tfidfv.transform(df_courses['Course Text'])

# Menampilkan fitur (kata unik)
print("Fitur (Kata unik):")
print(tfidfv.get_feature_names_out())

# Menampilkan matriks fitur dalam bentuk array
print("\nMatriks TF-IDF:")
print(course_vectors.toarray())


## S-BERT

In [ ]:
# Inisialisasi Model S-BERT
model_sbert = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

In [ ]:
# Encode text job menjadi vektor dengan S-BERT
job_sbert = model_sbert.encode(df_jobs['Job Text'].tolist(), show_progress_bar=True)

# Menampilkan matriks fitur dalam bentuk array
print("\nMatriks Job BERT:")
print(job_sbert)

In [ ]:
# Encode text course menjadi vektor dengan S-BERT
course_sbert = model_sbert.encode(df_courses['Course Text'].tolist(), show_progress_bar=True)

# Menampilkan matriks fitur dalam bentuk array
print("\nMatriks Course BERT:")
print(course_sbert)

## STEP 4: Similarity Mapping



In [ ]:
# Perhitungan Cosine Similarity dengan TF-IDF
similarity_tfidf = cosine_similarity(job_vectors, course_vectors)

# Matrix hasil similaritas vector
print("Matrix Cosine Similarity dengan TF-IDF:")
print(similarity_tfidf)

In [ ]:
# Perhitungan Cosine Similarity dengan S-BERT
similarity_sbert = cosine_similarity(job_sbert, course_sbert)

print("Matrix Cosine Similarity dengan S-BERT:")
print(similarity_sbert)

## STEP 5: Membangun Recommender System

In [ ]:
# Similarity Mapping tahap persiapan data ---
df_jobs_for_recommendation = df_jobs[['Job Title', 'Company', 'Job Description', 'Qualification', 'Benefit', 'Job Text']]
df_jobs_for_recommendation.head()

In [ ]:
# Recommendation Modelling: Fungsi Umum untuk Kedua Model ---
def get_job_recommendations(course_description, model_type, num_recommendations=5):
    if model_type == 'tfidf':
        course_vector = tfidfv.transform([course_description])
        similarity_matrix = cosine_similarity(course_vector, job_vectors)
    elif model_type == 'sbert':
        course_vector = model_sbert.encode([course_description])
        similarity_matrix = cosine_similarity(course_vector, job_sbert)
    else:
        raise ValueError("Invalid model_type. Choose 'tfidf' or 'sbert'.")

    # Flatten the similarity_matrix before getting top indices
    similarity_scores = similarity_matrix[0]

    # Get indices of top recommendations
    top_indices = similarity_scores.argsort()[-num_recommendations:][::-1]

    # Filter out recommendations with 0 or negative similarity score
    top_indices = [idx for idx in top_indices if similarity_scores[idx] > 0]


    if not top_indices:
        print("Peringatan: Tidak ada rekomendasi pekerjaan yang kuat (skor similaritas semuanya nol atau tidak ada).")
        return pd.DataFrame()

    recommended_jobs = df_jobs.iloc[top_indices].copy()
    recommended_jobs['Similarity Score'] = similarity_scores[top_indices]

    # Urutkan berdasarkan Similarity_Score secara menurun
    recommended_jobs = recommended_jobs.sort_values(by='Similarity Score', ascending=False)

    return recommended_jobs[['Job Title', 'Company', 'Job Description', 'Qualification', 'Benefit', 'Similarity Score']]

In [ ]:
# Recommendation Modelling: Contoh Penggunaan dan Perbandingan
test_course_desc = "Saya baru saja menyelesaikan kursus mendalam tentang UI/UX Designer, saya memahami dengan baik metode design thinking"

print(f"\nQuery: {test_course_desc}")

# Rekomendasi menggunakan TF-IDF
print("\n--- Rekomendasi dengan TF-IDF ---")
recommended_jobs_tfidf = get_job_recommendations(
    test_course_desc,
    model_type='tfidf',
    num_recommendations=5
)
if not recommended_jobs_tfidf.empty:
    print(recommended_jobs_tfidf)
else:
    print("Tidak ada rekomendasi TF-IDF ditemukan.")

# Rekomendasi menggunakan S-BERT
print("\n\n\n--- Rekomendasi dengan S-BERT ---")
recommended_jobs_sbert = get_job_recommendations(
    test_course_desc,
    model_type='sbert',
    num_recommendations=5
)
if not recommended_jobs_sbert.empty:
    print(recommended_jobs_sbert)
else:
    print("Tidak ada rekomendasi S-BERT ditemukan.")

In [ ]:
# Uji sistem rekomendasi dengan 5 data pertama

example_recommendations = {}

for i, course_name in enumerate(df_courses['Title'].tolist()[:5]):
    recs = get_job_recommendations(
        course_name,
        model_type='sbert',
        num_recommendations=10  # Total jumlah rekomendasi per query
    )
    print(f"\n{'='*100}")
    print(f"Query [{i+1}]: {course_name}")
    print(f"{'-'*100}")

    if not recs.empty:
        example_recommendations[f"{i+1}"] = {
            "query_text": course_name,
            "recommendations": recs.to_dict(orient='records')
        }

        for j, (_, row) in enumerate(recs.iterrows(), start=1):
            print(f"{j}. {row['Job Title']} at {row['Company']}")
            print(f"   Score: {row['Similarity Score']:.4f}")
    else:
        example_recommendations[f"{i+1}"] = {
            "query_text": course_name,
            "recommendations": []
        }
        print("No recommendations found.")

In [ ]:
# Blok kode ini tidak jadi dijalankan karena model S-BERT memiliki hasil rekomendasi yang lebih baik daripada TF-IDF

# # Simpan hasil TF-IDF ke file JSON
# recommendations_tfidf_filename = 'recommendations_tfidf.json'
# with open(recommendations_tfidf_filename, 'w', encoding='utf-8') as f:
#     json.dump(example_recommendations, f, ensure_ascii=False, indent=4)
# print(f"\n? Hasil rekomendasi TF-IDF disimpan ke '{recommendations_tfidf_filename}'")

In [ ]:
# Simpan hasil S-BERT ke file JSON
recommendations_sbert_filename = 'recommendations_sbert.json'
with open(recommendations_sbert_filename, 'w', encoding='utf-8') as f:
    json.dump(example_recommendations, f, ensure_ascii=False, indent=4)
print(f"\n? Hasil rekomendasi S-BERT disimpan ke '{recommendations_sbert_filename}'")

In [ ]:
# ? Simpan Model, Embeddings, dan Data
model_sbert.save(os.path.join(MODEL_DIR, 'sbert_model_dir'))

with open(os.path.join(MODEL_DIR, 'sbert_embeddings.pkl'), 'wb') as f:
    pickle.dump(job_sbert, f)

with open(os.path.join(MODEL_DIR, 'df_for_recommendation.pkl'), 'wb') as f:
    pickle.dump(df_jobs_for_recommendation, f)

print("? Semua model, embeddings, dan data berhasil disimpan di folder 'recommender_models'.")